# Siamese BiLSTM — Contrastive Retrieval

Neculoiu et al. (2016) Siamese Recurrent Network for learning text similarity.
Uses word2vec embedding, 4 BiLSTM layers, contrastive loss, Adam optimizer.
Trains on (question, answer) pairs with 4:1 negative ratio.

In [ ]:
!git clone https://github.com/hyperformancelabs/vnlegal-rag.git
%cd vnlegal-rag
!cat data/word2vec/word2vec_part_* > data/word2vec/merged.zip
!7z x data/word2vec/merged.zip -odata/word2vec/ -y

In [33]:
!git fetch origin
!git reset --hard origin/$(git branch --show-current)
!git clean -fd

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.59 KiB | 1.59 MiB/s, done.
From https://github.com/hyperformancelabs/vnlegal-rag
   5ac8010..f56ad71  master     -> origin/master
HEAD is now at f56ad71 fix: siamese vocab min_freq=1 (4668→6660, retains 30% more tokens)
Removing experiments/siamese_bilstm_artifacts/


In [1]:
%cd vnlegal-rag

/content/vnlegal-rag


In [2]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

from src.tokenizer import build_vocab, build_embedding_matrix, encode_text, simple_tokenize, PAD_TOKEN, UNK_TOKEN
from src.models.siamese_bilstm import SiameseBiLSTM, ContrastiveLoss

In [3]:
# Device and seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [4]:
# Data and artifact paths
DATA_DIR = Path('data/data_ready_k4')
ARTIFACT_DIR = Path('experiments/siamese_bilstm_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
W2V_PATH = Path('data/word2vec/word2vec_vi_syllables_300dims.txt')
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

DATA_DIR = data/data_ready_k4
ARTIFACT_DIR = experiments/siamese_bilstm_artifacts


In [5]:
# Load QA splits + corpus
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='	')
qa_val   = pd.read_csv(DATA_DIR / 'qa_val.csv',   sep='	')
qa_test  = pd.read_csv(DATA_DIR / 'qa_test.csv',  sep='	')
corpus   = pd.read_csv(DATA_DIR / 'corpus_full.csv', sep='	')

print('train:', qa_train.shape, 'val:', qa_val.shape, 'test:', qa_test.shape)
print('corpus:', corpus.shape)

train: (23408, 12) val: (2902, 12) test: (2832, 12)
corpus: (9582, 5)


In [6]:
# ── Siamese BiLSTM hyperparams (Neculoiu et al. 2016) ──
MAX_LEN = 256      # questions p95=90, answers p95=370, articles p95=740 — balance

# Architecture
NUM_BILSTM_LAYERS = 4
HIDDEN_DIM  = 64       # Per direction; output = 128-d after concat
DENSE_DIM   = 128      # Final projection after mean-pool

# Regularization
DROPOUT_RECURRENT = 0.2
DROPOUT_INTER     = 0.4
DROPOUT_OUT       = 0.4

# Contrastive loss (BCE mode avoids dead zone)
CONTRASTIVE_MARGIN  = 0.5
USE_BCE              = True  # BCE on cosine → always has gradient
NEGATIVE_RATIO      = 4
POSITIVE_LOSS_SCALE = 0.25

# Training
BATCH_SIZE = 256
ADAM_LR    = 0.001
GRAD_CLIP  = 5.0
EPOCHS     = 30
PATIENCE   = 5

In [7]:
# Build vocab from all texts (QA + corpus)
all_texts = pd.concat([
    qa_train['question'], qa_val['question'], qa_test['question'],
    qa_train['answer'],   qa_val['answer'],   qa_test['answer'],
    corpus['article_content'],
], ignore_index=True).tolist()

stoi = build_vocab(all_texts, max_vocab=100_000, min_freq=1)
pad_idx = stoi[PAD_TOKEN]
print(f'Vocab size: {len(stoi)}')

Vocab size: 6761


In [8]:
# Build pretrained embedding matrix
embedding_weight, hits = build_embedding_matrix(stoi, embed_dim=300, w2v_path=W2V_PATH)
print(f'Embedding shape: {embedding_weight.shape} | w2v hits: {hits}/{len(stoi) - 2}')
embedding_weight = torch.tensor(embedding_weight, dtype=torch.float32)

Embedding shape: torch.Size([6761, 300]) | w2v hits: 4027/6759


/tmp/ipykernel_16907/1362131438.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embedding_weight = torch.tensor(embedding_weight, dtype=torch.float32)


In [9]:
# Tokenize questions and answers
train_q_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_train['question']])
train_a_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_train['answer']])
val_q_ids   = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_val['question']])
val_a_ids   = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_val['answer']])
test_q_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_test['question']])
test_a_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_test['answer']])

# Corpus for retrieval eval
corpus_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in corpus['article_content']])
corpus_test_df = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='	')
corpus_test_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in corpus_test_df['article_content']])

# Token length stats
q_lens = [len(simple_tokenize(str(t))) for t in qa_train['question']]
a_lens = [len(simple_tokenize(str(t))) for t in qa_train['answer']]
c_lens = [len(simple_tokenize(str(t))) for t in corpus['article_content']]
print(f'Question: max={max(q_lens)} p95={np.percentile(q_lens, 95):.0f}')
print(f'Answer:   max={max(a_lens)} p95={np.percentile(a_lens, 95):.0f}')
print(f'Corpus:   max={max(c_lens)} p95={np.percentile(c_lens, 95):.0f}')
pct = lambda L: 100 * sum(1 for x in L if x > MAX_LEN) / max(len(L), 1)
print(f'Truncated at {MAX_LEN}: q={pct(q_lens):.1f}% a={pct(a_lens):.1f}% c={pct(c_lens):.1f}%')

Question: max=209 p95=90
Answer:   max=989 p95=368
Corpus:   max=92290 p95=743
Truncated at 256: q=0.0% a=17.3% c=35.8%


In [10]:
# Pair dataset with on-the-fly negative sampling
class SiamesePairDataset(Dataset):
    def __init__(self, q_ids, a_ids, neg_ratio=4):
        self.q_ids, self.a_ids, self.n = q_ids, a_ids, len(q_ids)
        self.neg_ratio = neg_ratio

    def __len__(self):
        return self.n * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        if idx < self.n:
            i = idx
            label = 1
        else:
            i = random.randrange(self.n)
            j = random.randrange(self.n)
            while j == i:
                j = random.randrange(self.n)
            label = 0
            return (torch.tensor(self.q_ids[i]), torch.tensor(self.a_ids[j]),
                    torch.tensor(label, dtype=torch.float32))
        return (torch.tensor(self.q_ids[i]), torch.tensor(self.a_ids[i]),
                torch.tensor(label, dtype=torch.float32))

In [11]:
# DataLoaders
import sys
nw = 0 if sys.platform == 'win32' else 2
pin = torch.cuda.is_available()

train_ds = SiamesePairDataset(train_q_ids, train_a_ids, neg_ratio=NEGATIVE_RATIO)
val_ds   = SiamesePairDataset(val_q_ids,   val_a_ids,   neg_ratio=NEGATIVE_RATIO)
test_ds  = SiamesePairDataset(test_q_ids,  test_a_ids,  neg_ratio=NEGATIVE_RATIO)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=pin)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)

print(f'Train: {len(train_ds):,} pairs ({train_ds.n:,} pos + {train_ds.n * NEGATIVE_RATIO:,} neg)')
print(f'Val:   {len(val_ds):,} pairs')
print(f'Test:  {len(test_ds):,} pairs')

Train: 117,040 pairs (23,408 pos + 93,632 neg)
Val:   14,510 pairs
Test:  14,160 pairs


In [12]:
# Build Siamese BiLSTM with pretrained word2vec embedding
model = SiameseBiLSTM(
    vocab_size=len(stoi),
    embedding_weight=embedding_weight,
    freeze_embedding=False,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_BILSTM_LAYERS,
    dense_dim=DENSE_DIM,
    dropout_recurrent=DROPOUT_RECURRENT,
    dropout_inter=DROPOUT_INTER,
    dropout_out=DROPOUT_OUT,
    pad_idx=pad_idx,
)
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Params: {total_params:,}')

Params: 2,530,188


In [13]:
# Contrastive loss + Adam (paper §3.1, §4)
criterion = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, positive_scale=POSITIVE_LOSS_SCALE, use_bce=USE_BCE)
optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

In [14]:
# Training loop
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for q, a, y in loader:
        q, a, y = q.to(device), a.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            cos = model(q, a)
            loss = criterion(cos, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * q.size(0)
    return total_loss / len(loader.dataset)

In [15]:
# Pair evaluation
def evaluate_pairs(model, loader):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for q, a, y in loader:
            q, a, y = q.to(device), a.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                cos = model(q, a)
                loss = criterion(cos, y)
            total_loss += loss.item() * q.size(0)
            pred = (torch.sigmoid(cos * 5.0) >= 0.5).float()  # BCE-scale aligned
            y_true.extend(y.cpu().tolist())
            y_pred.extend(pred.cpu().tolist())
    acc = sum(1 for t, p in zip(y_true, y_pred) if t == p) / max(len(y_true), 1)
    pos_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1) / max(sum(y_true), 1)
    neg_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0) / max(len(y_true) - sum(y_true), 1)
    return {'loss': total_loss / len(loader.dataset), 'acc': acc,
            'pos_acc': pos_acc, 'neg_acc': neg_acc}

In [16]:
# Retrieval evaluation: encode all → rank → MRR/mAP/recall@k
def retrieval_eval(model, q_ids, a_ids, corpus_ids):
    model.eval()
    bs = 256
    with torch.no_grad():
        q_embs = torch.cat([model.encoder(torch.tensor(q_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(q_ids), bs)], dim=0)
        c_embs = torch.cat([model.encoder(torch.tensor(corpus_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(corpus_ids), bs)], dim=0)
        a_embs = torch.cat([model.encoder(torch.tensor(a_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(a_ids), bs)], dim=0)
    sim = F.normalize(q_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T
    a_sim = F.normalize(a_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T
    gt = a_sim.argmax(dim=1).numpy()
    ranks = np.array([(sim[i] > sim[i, gt[i]]).sum() + 1 for i in range(len(q_ids))])
    return {
        'mrr': (1.0 / ranks).mean(),
        'map': (1.0 / ranks).mean(),
        **{f'recall@{k}': (ranks <= k).mean() for k in [1, 5, 10, 20]},
        'mean_rank': ranks.mean(), 'median_rank': np.median(ranks),
    }

In [ ]:
# Train with early stopping on val loss
best_path = ARTIFACT_DIR / 'siamese_bilstm_best.pt'
best_val_loss = float('inf')
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val = evaluate_pairs(model, val_loader)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val['loss'],
                    'val_acc': val['acc'], 'val_pos_acc': val['pos_acc'], 'val_neg_acc': val['neg_acc']})

    print(f'Epoch {epoch:2d} | train_loss: {train_loss:.4f} | val_loss: {val["loss"]:.4f} | '
          f'acc: {val["acc"]:.4f} | pos: {val["pos_acc"]:.4f} | neg: {val["neg_acc"]:.4f}')

    if val['loss'] < best_val_loss:
        best_val_loss = val['loss']
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
        print(f'  ✓ saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

Epoch  1 | train_loss: 0.0000 | val_loss: 0.0000 | acc: 0.2000 | pos: 1.0000 | neg: 0.0000
  ✓ saved (val_loss=0.0000)
Epoch  2 | train_loss: 0.0000 | val_loss: 0.0000 | acc: 0.2000 | pos: 1.0000 | neg: 0.0000
  ✓ saved (val_loss=0.0000)
Epoch  3 | train_loss: 0.0000 | val_loss: 0.0000 | acc: 0.2000 | pos: 1.0000 | neg: 0.0000
  ✓ saved (val_loss=0.0000)
Epoch  4 | train_loss: 0.0000 | val_loss: 0.0000 | acc: 0.2000 | pos: 1.0000 | neg: 0.0000
  ✓ saved (val_loss=0.0000)
Epoch  5 | train_loss: 0.0000 | val_loss: 0.0000 | acc: 0.2000 | pos: 1.0000 | neg: 0.0000
  ✓ saved (val_loss=0.0000)


In [ ]:
# Test set pair evaluation
model.load_state_dict(torch.load(best_path, map_location=device))
test = evaluate_pairs(model, test_loader)
print('--- Test Pair Results ---')
print(f'Accuracy:  {test["acc"]:.4f}')
print(f'Pos Acc:   {test["pos_acc"]:.4f}')
print(f'Neg Acc:   {test["neg_acc"]:.4f}')

In [ ]:
# Full retrieval evaluation
ret = retrieval_eval(model, test_q_ids, test_a_ids, corpus_test_ids)
print('--- Retrieval Results ---')
for k in ['mrr', 'map']:
    print(f'{k.upper():>11}: {ret[k]:.4f}')
for k in [1, 5, 10, 20]:
    print(f'  Recall@{k:>2}: {ret[f"recall@{k}"]:.4f}')
print(f'  Mean rank: {ret["mean_rank"]:.0f}')

In [ ]:
# Save artifacts
torch.save(model.encoder.state_dict(), ARTIFACT_DIR / 'encoder.pt')
torch.save(stoi, ARTIFACT_DIR / 'stoi.pt')

metadata = {
    'model': 'SiameseBiLSTM', 'paper': 'Neculoiu et al. 2016',
    'max_len': MAX_LEN, 'vocab_size': len(stoi), 'embed_dim': 300,
    'num_layers': NUM_BILSTM_LAYERS, 'hidden_dim': HIDDEN_DIM, 'dense_dim': DENSE_DIM,
    'dropout_recurrent': DROPOUT_RECURRENT, 'dropout_inter': DROPOUT_INTER,
    'dropout_out': DROPOUT_OUT, 'margin': CONTRASTIVE_MARGIN,
    'batch_size': BATCH_SIZE, 'adam_lr': ADAM_LR, 'grad_clip': GRAD_CLIP,
    'best_val_loss': best_val_loss, 'test_pair_acc': test['acc'],
    'test_retrieval': ret, 'history': history,
}
with open(ARTIFACT_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('Saved to', ARTIFACT_DIR)

In [ ]:
# Download artifacts (Colab)
import shutil
from google.colab import files
shutil.make_archive(str(ARTIFACT_DIR.parent / 'siamese_bilstm_export'), 'zip', ARTIFACT_DIR)
files.download(str(ARTIFACT_DIR.parent / 'siamese_bilstm_export.zip'))